# 🚀 Experiment B: 4-Bit NF4 QLoRA Fine-Tuning Pipeline (NVIDIA CUDA)
**Objective:** Fine-tune a 7B/8B parameter LLM on a single GPU using **BitsAndBytes 4-bit NormalFloat4 (NF4) quantization**, **Double Quantization**, and **PEFT/LoRA adapters** to achieve $>90\%$ VRAM reduction.

In [ ]:
# 1. Install Dependencies
!pip install -q torch transformers peft trl bitsandbytes accelerate datasets wandb

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # or "meta-llama/Meta-Llama-3-8B-Instruct"
OUTPUT_DIR = "./qlora_7b_adapter"

print(f"GPU Available: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
# 2. Configure BitsAndBytes 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 3. Load Base Model in 4-bit
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)
print("✅ 7B Base Model Loaded in 4-Bit NF4!")

In [ ]:
# 4. Inject LoRA PEFT Adapters (r=16, alpha=32)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

qlora_model = get_peft_model(model, peft_config)
qlora_model.print_trainable_parameters()

In [ ]:
# 5. Execute 4-bit QLoRA Training
# Sample clinical data
dataset = Dataset.from_list([
    {"text": "<|im_start|>user\nA 62-year-old male with CKD (eGFR 26) has diabetes. Which drug is contraindicated?<|im_end|>\n<|im_start|>assistant\nMetformin is contraindicated due to lactic acidosis risk. Recommend SGLT2i or GLP-1 RA.<|im_end|>"}
])

training_args = SFTConfig(
    output_dir="./qlora_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="text",
    max_length=512
)

trainer = SFTTrainer(
    model=qlora_model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
print(f"✅ QLoRA Adapter saved to {OUTPUT_DIR}")